# ACIS Insurance Risk Analytics: Hypothesis Testing

This notebook validates whether observed risk and profitability differences are statistically meaningful for ACIS. The focus is on province, zip code, margin, and gender hypotheses required by the assignment.

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data_loader import load_insurance_data, summarize_dataset
from src.hypothesis_tests import (
    add_margin_and_claim_flag,
    build_hypothesis_summary,
    chi_square_test,
    claim_rate_z_test,
    independent_t_test,
    top_two_groups,
)

ALPHA = 0.05
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:.4f}".format)

## Load Cleaned Data

The analysis uses the DVC-generated cleaned dataset. `HasClaim` and `Margin` are derived for the statistical tests.

In [2]:
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "insurance_data_cleaned.csv"

df = load_insurance_data(DATA_PATH)
df = add_margin_and_claim_flag(df)

summarize_dataset(df)

{'rows': 10000,
 'columns': 23,
 'total_cells': 230000,
 'missing_cells': 0,
 'missing_percentage': 0.0,
 'duplicate_rows': 0,
 'duplicate_percentage': 0.0,
 'column_types': {'CustomerID': 'str',
  'Age': 'int64',
  'Gender': 'str',
  'Province': 'str',
  'VehicleType': 'str',
  'AnnualIncome': 'int64',
  'RiskScore': 'int64',
  'AnnualPremium': 'int64',
  'Deductible': 'int64',
  'NCD': 'int64',
  'PastClaims': 'int64',
  'Claimed': 'bool',
  'ClaimAmount': 'float64',
  'TotalPremium': 'int64',
  'TotalClaims': 'float64',
  'CoverType': 'str',
  'AutoMake': 'str',
  'VehicleModel': 'str',
  'CustomValueEstimate': 'int64',
  'ZipCode': 'int64',
  'TransactionDate': 'str',
  'Margin': 'float64',
  'HasClaim': 'int64'}}

## Hypotheses, KPIs, and Tests

| Business question | KPI | Test |
| --- | --- | --- |
| Are there risk differences across provinces? | Claim Frequency (`HasClaim`) | Chi-square test |
| Are there risk differences between high-volume provinces? | Claim Frequency (`HasClaim`) | Two-proportion z-test |
| Is there a margin difference between high-volume zip codes? | Margin | Welch's t-test |
| Is there a risk difference between Women and Men? | Claim Frequency (`HasClaim`) | Chi-square test / z-test |

Decision rule: reject the null hypothesis when `p-value < 0.05`.

In [3]:
province_col = "Province"
gender_col = "Gender"
zip_col = "ZipCode"
claims_col = "TotalClaims"

province_control, province_test = top_two_groups(df, province_col)
zip_control, zip_test = top_two_groups(df, zip_col)
gender_control, gender_test = top_two_groups(df, gender_col)

province_control, province_test, zip_control, zip_test, gender_control, gender_test

('Addis Ababa', 'Oromia', 10004, 10002, 'Female', 'Male')

In [4]:
results = []

results.append(
    chi_square_test(
        df,
        group_col=province_col,
        outcome_col="HasClaim",
        hypothesis="H0: There are no claim-frequency differences across provinces.",
        kpi="Claim Frequency",
        alpha=ALPHA,
        rejected_message="Claim frequency differs by province; ACIS should review regional risk pricing.",
        not_rejected_message="No statistically significant province-level claim-frequency difference was detected.",
    )
)

results.append(
    claim_rate_z_test(
        df,
        group_col=province_col,
        claims_col=claims_col,
        control_group=province_control,
        test_group=province_test,
        hypothesis=f"H0: Claim frequency is equal between {province_control} and {province_test}.",
        alpha=ALPHA,
    )
)

results.append(
    independent_t_test(
        df,
        group_col=zip_col,
        value_col="Margin",
        control_group=zip_control,
        test_group=zip_test,
        hypothesis=f"H0: Average margin is equal between zip codes {zip_control} and {zip_test}.",
        kpi="Margin",
        alpha=ALPHA,
        rejected_message="Average margin differs by zip code; ACIS should review localized pricing and customer mix.",
        not_rejected_message="No statistically significant margin difference was detected between the selected zip codes.",
    )
)

results.append(
    chi_square_test(
        df,
        group_col=gender_col,
        outcome_col="HasClaim",
        hypothesis="H0: There is no claim-frequency difference between Women and Men.",
        kpi="Claim Frequency",
        alpha=ALPHA,
        rejected_message="Claim frequency differs by gender; ACIS should investigate exposure mix before business action.",
        not_rejected_message="No statistically significant gender-level claim-frequency difference was detected.",
    )
)

results.append(
    independent_t_test(
        df,
        group_col=gender_col,
        value_col="Margin",
        control_group=gender_control,
        test_group=gender_test,
        hypothesis=f"H0: Average margin is equal between {gender_control} and {gender_test}.",
        kpi="Margin",
        alpha=ALPHA,
        rejected_message="Average margin differs by gender; ACIS should investigate whether other factors explain the difference.",
        not_rejected_message="No statistically significant gender-level margin difference was detected.",
    )
)

summary = build_hypothesis_summary(results)
summary

,hypothesis,kpi,test_used,p_value,decision,business_interpretation,control_group,test_group,sample_size
0,H0: There are no claim-frequency differences a...,Claim Frequency,Chi-square test of independence,0.0761,Do not reject null,No statistically significant province-level cl...,NaN,NaN,10000
1,H0: Claim frequency is equal between Addis Aba...,Claim Frequency,Two-proportion z-test,0.7859,Do not reject null,Do not reject the null hypothesis. The data do...,Addis Ababa,Oromia,6013
2,H0: Average margin is equal between zip codes ...,Margin,Welch's t-test,0.2642,Do not reject null,No statistically significant margin difference...,10004,10002,1465
3,H0: There is no claim-frequency difference bet...,Claim Frequency,Chi-square test of independence,0.9638,Do not reject null,No statistically significant gender-level clai...,NaN,NaN,10000
4,H0: Average margin is equal between Female and...,Margin,Welch's t-test,0.9847,Do not reject null,No statistically significant gender-level marg...,Female,Male,10000


## Save Report-Ready Results

The summary table is saved to `reports/hypothesis_test_results.csv` so it can be reused directly in the final report.

In [5]:
REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_DIR.mkdir(exist_ok=True)

summary.to_csv(REPORTS_DIR / "hypothesis_test_results.csv", index=False)

rejected_hypotheses = summary[summary["decision"] == "Reject null"]
rejected_hypotheses[["hypothesis", "business_interpretation"]]

,hypothesis,business_interpretation


## Leadership Interpretation

For rejected hypotheses, ACIS can treat the tested segment difference as statistically supported and investigate pricing or underwriting action. For hypotheses that are not rejected, ACIS should avoid making pricing changes based on that segment alone and instead continue monitoring the segment with more data or richer controls.